# Analyzing DHS microdata for India

India 2015-2016 - J:\DATA\DHS_PROG_DHS\IND\2015_2016

More recent is available, but probably weird due to COVID

## Documentation

DHS 7 recode manual for variable definitions: https://www.dhsprogram.com/pubs/pdf/DHSG4/Recode7_DHS_10Sep2018_DHSG4.pdf.
However, it does not state which variables are in which file.
The `.MAP` files alongside each data file list the variables in it and what they mean.

Hemoglobin variables of interest: 
- HA53: Hemoglobin level in g/dl with 1 implied dcimal
- HA54: Currently pregnant
- HA55 Result of Hemoglobin measuring.
- HA56 Hemoglobin level adjusted by altitude in g/dl with 1 implied decimal. 

Wealth index variables of interest: 
- HV270: The wealth index is a composite measure of a household's cumulative living standard.
The wealth index is calculated using easy-to-collect data on a household’s ownership of
selected assets, such as televisions and bicycles; materials used for housing construction; and
types of water access and sanitation facilities.
Generated with a statistical procedure known as principal components analysis, the wealth
index places individual households on a continuous scale of relative wealth. DHS separates
all interviewed households into five wealth quintiles to compare the influence of wealth on
various population, health and nutrition indicators. The wealth index is presented in the DHS
Final Reports and survey datasets as a background characteristic
- HV271: Wealth index factor score (5 decimals) 

Pregnancy variables of interest: 
- HML18: Pregnancy status from individual questionnaire. For complete woman’s interviews this is
taken from V213. For incomplete woman's interview with anemia testing the pregnancy
status is taken from this section.
BASE: Women with a completed individual questionnaire or when available information
from the anemia testing section.

List of datasets: https://www.dhsprogram.com/data/dataset/Nigeria_Standard-DHS_2018.cfm?flag=1

Instructions on how to calculate everything can be found at: https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-7_v2.pdf

In [ ]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

## Load data, name columns

In [ ]:
directory = '/snfs1/DATA/DHS_PROG_DHS/IND/2015_2016/'

### WRA

In [ ]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
raw_wra_data = pd.read_stata(directory + 'IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA', columns=wra_columns.keys())

In [ ]:
wra_data =  raw_wra_data.copy()
wra_data

In [ ]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [ ]:
def recode_wealth_quintile(df):
    return df.map({
        "poorest": "lowest",
        "poorer": "second",
        "middle": "middle",
        "richer": "fourth",
        "richest": "highest",
    })

In [ ]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [ ]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Births

In [ ]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    "s220a": "duration_of_pregnancy",
}
birth_data =  pd.read_stata(directory + 'IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA', columns=birth_columns.keys())
birth_data

In [ ]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

### Household members

In [ ]:
%%time

hhm_columns = {
    "hv001": "cluster_number",
    "hv002": "household_number",
    "hv005": "weight",
    "hv008": "date_of_interview",
    "hvidx": "line_number",
    "hml18": "currently_pregnant",
    "hv105": "age",
    "hv104": "sex",
    "ha0": "index_to_household",
    "ha1": "age_hemoglobin",
    "hv270": "wealth_quintile",
    "ha53": "hemoglobin_raw",
    "ha56": "hemoglobin_adjusted",
    "ha57": "anemia",
}
hhm_data =  pd.read_stata(directory + 'IND_DHS7_2015_2016_HHM_IAPR74FL_Y2018M12D06.DTA', columns=hhm_columns.keys())
hhm_data

In [ ]:
hhm_data = hhm_data[hhm_columns.keys()].rename(columns=hhm_columns)

In [ ]:
hhm_data["age"] = hhm_data.age.replace({"95+": 95, "don't know": np.nan}).astype(float)

In [ ]:
# Interesting -- sometimes age is quite off.
hhm_data.loc[(hhm_data.age - hhm_data.age_hemoglobin).sort_values().index]

In [ ]:
(hhm_data.age - hhm_data.age_hemoglobin).describe()

In [ ]:
hhm_data["wealth_quintile"] = recode_wealth_quintile(hhm_data.wealth_quintile)
hhm_data["weight"] = hhm_data.weight / 1_000_000

In [ ]:
for col in ["hemoglobin_raw", "hemoglobin_adjusted"]:
    hhm_data[col] = hhm_data[col].astype(str).replace({
        'not tested': np.nan,
        'not present': np.nan,
        'refused': np.nan,
        'other': np.nan
    }).astype(float)

### Adult mortality

Adult mortality included in HH for India (recent household members who died).

In [ ]:
%%time

household_columns = {
    "hv005": "weight",
    "sh70": "any_died",
    "sh71": "num_died",
    "hv270": "wealth_quintile",
}
MAX_NUM_DEATHS = 5
death_columns = {
    "sh73": "sex",
    "sh74u": "age_at_death_unit",
    "sh74n": "age_at_death",
    "sh75m": "month_of_death",
    "sh75y": "year_of_death",
    "sh76": "death_violence_or_accident",
    "sh77": "death_during_pregnancy_or_childbirth",
}
columns = list(household_columns.keys())
for death_num in range(1, MAX_NUM_DEATHS + 1):
    columns += [c + '_' + str(death_num) for c in death_columns.keys()]

adult_mortality_data = pd.read_stata(directory + 'IND_DHS7_2015_2016_HH_IAHR74FL_Y2018M12D06.DTA', columns=columns)
adult_mortality_data

In [ ]:
# inspired by https://stackoverflow.com/a/67393747/
adult_mortality_data_reshaped = adult_mortality_data[[c for c in adult_mortality_data.columns if c.split('_')[0] in death_columns.keys()]].copy()
adult_mortality_data_reshaped.columns = adult_mortality_data_reshaped.columns.str.split("_", expand = True)
adult_mortality_data_reshaped

In [ ]:
adult_mortality_data_reshaped[list(household_columns.keys())] = adult_mortality_data[list(household_columns.keys())]
adult_mortality_data_reshaped

In [ ]:
# Get a row per death
adult_mortality_data_reshaped = adult_mortality_data_reshaped.set_index(list(household_columns.keys())).swaplevel(axis=1).stack(0).reset_index().drop(columns=[f"level_{len(household_columns)}"])
adult_mortality_data_reshaped

In [ ]:
adult_mortality_data = (
    adult_mortality_data_reshaped[list(household_columns.keys()) + list(death_columns.keys())]
        .rename(columns=household_columns)
        .rename(columns=death_columns)
)
adult_mortality_data["wealth_quintile"] = recode_wealth_quintile(adult_mortality_data.wealth_quintile)
adult_mortality_data["weight"] = adult_mortality_data.weight / 1_000_000
adult_mortality_data

## WRA

### Hemoglobin among pregnancies

In [ ]:
id_columns = ["cluster_number", "household_number", "line_number"]
other_overlapping_columns = (set(wra_data.columns) & set(hhm_data.columns)) - set(id_columns) - {'weight'}
other_overlapping_columns

In [ ]:
wra_hhm_joined = wra_data.merge(
    hhm_data.drop(columns=["weight"]),
    on=id_columns,
    suffixes=("_wra", "_hhm"),
    how="left",
)
wra_hhm_joined

In [ ]:
for col in other_overlapping_columns:
    assert (wra_hhm_joined[f'{col}_wra'] == wra_hhm_joined[f'{col}_hhm']).all()
    wra_hhm_joined[col] = wra_hhm_joined[f'{col}_wra']
    wra_hhm_joined = wra_hhm_joined.drop(columns=[f'{col}_wra', f'{col}_hhm'])

In [ ]:
col

In [ ]:
wra_hhm_joined.currently_pregnant.value_counts(dropna=False)

In [ ]:
pregnant_data = wra_hhm_joined[wra_hhm_joined.currently_pregnant == 'pregnant'].copy()
pregnant_data

In [ ]:
# https://stackoverflow.com/a/2415343/ with some tweaks
def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they 
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    is_nan = np.isnan(values)
    values = values[~is_nan]
    weights = weights[~is_nan]
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values-average)**2, weights=weights)
    return pd.Series({
        'mean': average,
        'sd': np.sqrt(variance),
        # https://ngreifer.github.io/WeightIt/reference/ESS.html
        'effective_sample_size': (weights.sum() ** 2) / (weights ** 2).sum()
    })

In [ ]:
# Matches table 10.21.1
pregnant_data[pregnant_data.anemia.notnull()].weight.sum()

In [ ]:
# Within rounding error of table 10.21.1 value
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia == 'severe',
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

In [ ]:
# Within rounding error of table 10.21.1 value for any anemia
weighted_avg_and_std(
    pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(['severe', 'moderate', 'mild']),
    pregnant_data[pregnant_data.anemia.notnull()].weight,
)

In [ ]:
assert (
    (pregnant_data[pregnant_data.anemia.notnull()].anemia == 'severe') ==
    (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 70)
).all()

In [ ]:
assert (
    (pregnant_data[pregnant_data.anemia.notnull()].anemia.isin(['severe', 'moderate', 'mild'])) ==
    (pregnant_data[pregnant_data.anemia.notnull()].hemoglobin_adjusted < 110)
).all()

In [ ]:
age_bin_edges = [15, 25, 30, 50]
age_bin_edges

In [ ]:
pregnant_data["age_group"] = pd.IntervalIndex(pd.cut(pregnant_data.age_hemoglobin, age_bin_edges, right=False))

In [ ]:
# NOTE: We could use this; we just don't have the sample size for it in Nigeria
(
    pregnant_data.groupby(["age_group", "wealth_quintile"])
        .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
        .sort_index()
)

In [ ]:
hemoglobin_disparities = (
    pregnant_data.groupby(["wealth_quintile"])
        .apply(lambda df: weighted_avg_and_std(df.hemoglobin_adjusted, weights=df.weight))
        .sort_index()
)
hemoglobin_disparities

In [ ]:
hemoglobin_disparities = hemoglobin_disparities.reset_index()
hemoglobin_disparities["sex"] = "Female"
hemoglobin_disparities = hemoglobin_disparities.set_index(["sex", "wealth_quintile"])
hemoglobin_disparities

In [ ]:
pregnancy_sim_input_data_dir = '../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data'

In [ ]:
hemoglobin_disparities["mean"].rename("value").to_csv(f'{pregnancy_sim_input_data_dir}/mean_hemoglobin_disparities/india.csv')

In [ ]:
hemoglobin_disparities["sd"].rename("value").to_csv(f'{pregnancy_sim_input_data_dir}/sd_hemoglobin_disparities/india.csv')

### Wealth quintile probabilities

Intuitively, you might think these would be equal; but we are looking at a subpopulation (pregnancies) that skews poorer.

In [ ]:
assert pregnant_data.wealth_quintile.notnull().all()

In [ ]:
wealth_quintile_probabilities = []

for quintile in pregnant_data.wealth_quintile.unique():
    quintile_info = pd.DataFrame(weighted_avg_and_std(pregnant_data.wealth_quintile == quintile, pregnant_data.weight)).T
    quintile_info.insert(0, "wealth_quintile", quintile)
    wealth_quintile_probabilities.append(quintile_info)

wealth_quintile_probabilities = pd.concat(wealth_quintile_probabilities, ignore_index=True)
wealth_quintile_probabilities.sort_values('mean')

In [ ]:
wealth_quintile_probabilities['mean'].sum()

In [ ]:
wealth_quintile_probabilities = wealth_quintile_probabilities.set_index("wealth_quintile")["mean"].to_frame().T.reset_index(drop=True)
wealth_quintile_probabilities.columns.name = None
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities.insert(0, "sex", "Female")
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities.to_csv(f'{pregnancy_sim_input_data_dir}/wealth_quintile_probabilities/india.csv', index=False)

### Maternal mortality ratio

#### Maternal mortality rate

Not reported anywhere for India DHS, so we need to be extra careful since we can't cross-check.

In [ ]:
adult_mortality_data.death_during_pregnancy_or_childbirth.value_counts()

In [ ]:
adult_mortality_data.death_violence_or_accident.value_counts()

In [ ]:
hhm_data

In [ ]:
adult_mortality_data["age_at_death_years"] = (
    adult_mortality_data.age_at_death_unit.map({'year': 1, 'months': 1 / 12, 'days': 1 / 365.25}) *
    adult_mortality_data.age_at_death
)

In [ ]:
adult_mortality_data["adult_death"] = (
    (adult_mortality_data.age_at_death_years >= 15) &
    (adult_mortality_data.age_at_death_years < 50)
)

In [ ]:
adult_mortality_data["date_of_death"] = adult_mortality_data.year_of_death.replace({"don't know": np.nan}).astype(float) + (
    adult_mortality_data.month_of_death.map({
        'january': 1,
        'february': 2,
        'march': 3,
        'april': 4,
        'may': 5,
        'june': 6,
        'july': 7,
        'august': 8,
        'september': 9,
        'october': 10,
        'november': 11,
        'december': 12,
    }) - 0.5
) * (1 / 12)

In [ ]:
adult_mortality_data["date_of_birth"] = adult_mortality_data.date_of_death - adult_mortality_data.age_at_death_years
adult_mortality_data["exposure_start"] = np.maximum(adult_mortality_data.date_of_birth + 15, 2011.0)
adult_mortality_data["exposure_end"] = np.minimum(adult_mortality_data.date_of_birth + 50, adult_mortality_data.date_of_death)
adult_mortality_data["exposure"] = np.maximum(adult_mortality_data.exposure_end - adult_mortality_data.exposure_start, 0)
adult_mortality_data["weighted_exposure"] = adult_mortality_data.weight * adult_mortality_data.exposure
adult_mortality_data

In [ ]:
living_exposure_data = hhm_data.copy()
living_exposure_data["date_of_birth"] = (living_exposure_data.date_of_interview / 12) + 1900 - living_exposure_data.age
living_exposure_data["exposure_start"] = np.maximum(living_exposure_data.date_of_birth + 15, 2011.0)
living_exposure_data["exposure_end"] = np.minimum(living_exposure_data.date_of_birth + 50, (living_exposure_data.date_of_interview / 12) + 1900)
living_exposure_data["exposure"] = np.maximum(living_exposure_data.exposure_end - living_exposure_data.exposure_start, 0)
living_exposure_data["weighted_exposure"] = living_exposure_data.weight * living_exposure_data.exposure
living_exposure_data

In [ ]:
(adult_mortality_data.adult_death * adult_mortality_data.weight).sum()

In [ ]:
adult_mortality_data.weighted_exposure.sum()

In [ ]:
((adult_mortality_data.adult_death * adult_mortality_data.weight).sum()) / (adult_mortality_data.weighted_exposure.sum() + living_exposure_data.weighted_exposure.sum())

In [ ]:
adult_mortality_data

In [ ]:
adult_mortality_data["maternal_death"] = (
    adult_mortality_data.adult_death &
    (adult_mortality_data.death_during_pregnancy_or_childbirth == 'yes') &
    (adult_mortality_data.death_violence_or_accident != 'yes')
)

In [ ]:
(adult_mortality_data.maternal_death * adult_mortality_data.weight).sum()

In [ ]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (df.weighted_exposure).sum()

In [ ]:
maternal_mortality_data = pd.concat([
    adult_mortality_data[adult_mortality_data.sex == 'female'][["wealth_quintile", "maternal_death", "weight", "weighted_exposure"]],
    living_exposure_data[living_exposure_data.sex == 'female'][["wealth_quintile", "weight", "weighted_exposure"]].assign(maternal_death=False)
], ignore_index=True)

In [ ]:
maternal_mortality_rate(maternal_mortality_data)

In [ ]:
maternal_mortality_rates = maternal_mortality_data.groupby("wealth_quintile").apply(maternal_mortality_rate)
maternal_mortality_rates

#### General fertility rate

In [ ]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    ((fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1) &
    ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
)
fertility_event_data["weighted_birth_in_period"] = fertility_event_data.birth_in_period * fertility_event_data.weight

In [ ]:
fertility_event_data.weighted_birth_in_period.sum()

In [ ]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

In [ ]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(fertility_exposure_data.date_of_birth + 12 * 15, fertility_exposure_data.interview_date - 36) # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(fertility_exposure_data.interview_date - 1, fertility_exposure_data.date_of_birth + 12 * 45)
fertility_exposure_data["exposure"] = ((fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = fertility_exposure_data.exposure * fertility_exposure_data.weight

In [ ]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum() * 1_000 /
    (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

In [ ]:
maternal_disorders_incidence_disparities = (maternal_mortality_rates / gfr_by_wealth).rename("value").rename_axis("wealth_quintile").reset_index()
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

In [ ]:
maternal_disorders_incidence_disparities.to_csv(f'{pregnancy_sim_input_data_dir}/maternal_disorders_incidence_disparities/india.csv', index=False)

## LBWSG

### Birth weight

In [ ]:
birth_data["birth_weight_kilograms"] = birth_data.birth_weight_kilograms.replace({'not weighed at birth': np.nan, "don't know": np.nan}).astype(float)

In [ ]:
weighted_avg_and_std(birth_data.birth_weight_kilograms, birth_data.weight)

In [ ]:
birth_weight_disparities = (
    birth_data.groupby("wealth_quintile").apply(lambda df: weighted_avg_and_std(df.birth_weight_kilograms, df.weight))
)
birth_weight_disparities

In [ ]:
birth_weight_disparities = birth_weight_disparities["mean"].rename("value").reset_index()
birth_weight_disparities

In [ ]:
child_sim_input_data_dir = '../../0300_child_sim/src/vivarium_gates_lsff_by_wealth_quintile_child/data/raw_data'

birth_weight_disparities.to_csv(f'{child_sim_input_data_dir}/birth_weight_disparities/india.csv', index=False)

### Short gestation

All we have here is a self-reported duration of pregnancy.

In [ ]:
# Basically no difference in mean
birth_data.groupby("wealth_quintile").apply(lambda df: weighted_avg_and_std(df.duration_of_pregnancy, df.weight))

In [ ]:
birth_data["short_gestation"] = np.where(
    birth_data.duration_of_pregnancy.isnull(),
    np.nan,
    birth_data.duration_of_pregnancy < 9.0
)

In [ ]:
weighted_avg_and_std(birth_data.short_gestation, birth_data.weight)

In [ ]:
birth_data.groupby("wealth_quintile").apply(lambda df: weighted_avg_and_std(df.short_gestation, df.weight))

The trends in short gestation don't make intuitive sense (?), so we do not plan to use them in the sim.